# Random forest classifier tutorial

We train `mlpackage.supervised_learning.RandomForestClassifier` on **Iris**: bootstrap **DecisionTreeClassifier** learners, **majority vote** at prediction time, and optional **`max_features="sqrt"`** column subsampling per tree. Set **`numpy.random.seed`** before **`fit`** for reproducible bagging.

## Prerequisites and goals

**You will**
- Fit an ensemble with **`n_estimators`**, **`max_depth`**, and **`max_features`**.
- Evaluate **accuracy** (this class has no `score` method—compare `predict` to labels explicitly).
- Plot **multiclass regions** in 2D using a **two-feature** refit (visualization only).
- Sweep **`n_estimators`** on the same split.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `mlpackage`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from mlpackage.supervised_learning import RandomForestClassifier

## Step 1: Load Iris

In [ ]:
iris = load_iris()
X = np.asarray(iris.data, dtype=float)
y = np.asarray(iris.target, dtype=int)

feature_names = list(iris.feature_names)
target_names = list(iris.target_names)

print("X shape:", X.shape, "  y shape:", y.shape)
print("Classes:", {i: target_names[i] for i in range(3)})

## Step 2: Stratified train/test split

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape[0], "  Test:", X_test.shape[0])

## Step 3: Why we skip scaling here

Axis-aligned tree splits depend on **ordering** of feature values, not affine rescaling of a single coordinate in isolation. For apples-to-apples comparisons with distance-based models you might still standardize, but it is optional for random forests on Iris.

## Step 4: Fit `RandomForestClassifier`

Call **`np.random.seed`** immediately before **`fit`** so bootstrap and feature subsampling are reproducible.

In [ ]:
RNG = 42
N_TREES = 80
MAX_DEPTH = 6

np.random.seed(RNG)
rf = RandomForestClassifier(
    n_estimators=N_TREES,
    max_depth=MAX_DEPTH,
    max_features="sqrt",
)
rf.fit(X_train, y_train)
print("Fitted RandomForestClassifier with", N_TREES, "trees.")

## Step 5: Predict on the held-out test set

In [ ]:
y_pred = rf.predict(X_test)

print("First 12 test predictions:", y_pred[:12].tolist())
print("First 12 true labels:    ", y_test[:12].tolist())

## Step 6: Accuracy and per-class test performance

`RandomForestClassifier` exposes **`predict`** only—accuracy is the mean of elementwise equality.

In [ ]:
def accuracy(y_true: np.ndarray, y_hat: np.ndarray) -> float:
    return float(np.mean(y_true == y_hat))


y_hat_train = rf.predict(X_train)
print(f"Train accuracy: {accuracy(y_train, y_hat_train):.4f}")
print(f"Test accuracy : {accuracy(y_test, y_pred):.4f}")

for c in range(3):
    mask = y_test == c
    hits = int(np.sum((y_pred == y_test) & mask))
    tot = int(np.sum(mask))
    print(f"Test {target_names[c]}: {hits}/{tot} correct")

n_show = 10
display(
    pd.DataFrame(
        {
            "y_true": [target_names[t] for t in y_test[:n_show]],
            "y_pred": [target_names[t] for t in y_pred[:n_show]],
        }
    )
)

## Step 7: 2D decision regions (two-feature forest, test points)

The forest above uses **four** inputs. To mimic lecture slides we **refit** on **petal length** and **petal width** only (`max_features=None` so both columns are available to every split), build a mesh, and **`predict`** a class label per grid cell.

In [ ]:
i, j = 2, 3
X_train_2d = X_train[:, [i, j]]
X_test_2d = X_test[:, [i, j]]

np.random.seed(RNG)
viz = RandomForestClassifier(
    n_estimators=60,
    max_depth=10,
    max_features=None,
)
viz.fit(X_train_2d, y_train)

pad = 0.35
h = 0.02
x_min = float(min(X_train_2d[:, 0].min(), X_test_2d[:, 0].min()) - pad)
x_max = float(max(X_train_2d[:, 0].max(), X_test_2d[:, 0].max()) + pad)
y_min = float(min(X_train_2d[:, 1].min(), X_test_2d[:, 1].min()) - pad)
y_max = float(max(X_train_2d[:, 1].max(), X_test_2d[:, 1].max()) + pad)

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, h),
    np.arange(y_min, y_max, h),
)
grid = np.c_[xx.ravel(), yy.ravel()]
Z = viz.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(
    xx,
    yy,
    Z,
    levels=[-0.5, 0.5, 1.5, 2.5],
    colors=["#ffcccc", "#ccffcc", "#cce5ff"],
    alpha=0.9,
)

markers = ["o", "s", "^"]
colors_pt = ["crimson", "darkgreen", "navy"]
for c in range(3):
    m = y_test == c
    plt.scatter(
        X_test_2d[m, 0],
        X_test_2d[m, 1],
        c=colors_pt[c],
        marker=markers[c],
        s=65,
        edgecolors="black",
        linewidths=0.6,
        label=f"{target_names[c]} (test)",
        zorder=5,
    )

plt.xlabel(feature_names[i])
plt.ylabel(feature_names[j])
plt.title("Random forest — 2D class regions (petal features, test overlay)")
plt.legend(loc="upper left", fontsize=9)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.tight_layout()

_nb = Path("random_forest_classifier_tutorial.ipynb")
_out = (
    _nb.with_name("iris_random_forest_decision_regions.png")
    if _nb.is_file()
    else Path(
        "examples/supervised_learning/random_forest_classifier/iris_random_forest_decision_regions.png"
    )
)
_out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(_out, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_out.resolve()}")
print("2-feature test accuracy:", round(accuracy(y_test, viz.predict(X_test_2d)), 4))
plt.show()

## Step 8: Compare `n_estimators` (same `RNG`, `max_depth`, split)

Each row refits from scratch; **`np.random.seed(RNG)`** is reset before every **`fit`** so comparisons are apples-to-apples for randomness.

In [ ]:
sizes = [20, 40, 80, 120]
rows = []

for n in sizes:
    np.random.seed(RNG)
    model = RandomForestClassifier(
        n_estimators=n,
        max_depth=MAX_DEPTH,
        max_features="sqrt",
    )
    model.fit(X_train, y_train)
    rows.append(
        {
            "n_estimators": n,
            "train_acc": accuracy(y_train, model.predict(X_train)),
            "test_acc": accuracy(y_test, model.predict(X_test)),
        }
    )

display(pd.DataFrame(rows).round(4))